# F2 · Linear PCA against KernelPCA on nonlinear data (and what it costs)

**Type**: Compare · **Level**: ●●● challenge · **Time**: about 55 minutes · **extended path**

## Objective

Show that on nonlinear data (two concentric circles, two moons) linear PCA does not help a linear classifier, while `KernelPCA` with an RBF kernel makes the problem separable with a few components; choose components and `gamma` on a grid and measure how the computation time grows with the number of examples.

## Dataset

`make_circles(1000, noise=0.05, factor=0.4)` and `make_moons(1000, noise=0.15)`. Model: `LogisticRegression`; validation `StratifiedKFold(5, shuffle=True, random_state=0)`.

## Educational purpose

Understand what a **nonlinear** PCA does (the kernel trick: PCA in the feature space induced by the kernel) and what it costs: the kernel matrix is `n × n`, so the cost grows with the square (or more) of the number of examples. This is why KernelPCA is only used with small or medium datasets.

## Task

1. For each dataset compute `base` (accuracy of the logistic alone on the raw data) and `pca2` (`PCA(2)` → logistic).
2. For `KernelPCA(k, kernel="rbf", gamma=g)` → logistic, with `k` in `[2, 5, 10]` and `g` in `[1, 3, 10]`, compute the accuracy and save it in `grid` (dictionary `dataset → DataFrame` with `k` rows and `g` columns). Save `best` (dictionary dataset → tuple `(k, g)` with the highest accuracy; on ties, the smaller `k` and then the smaller `g`) and `min_k` (dictionary dataset → the smallest `k` for which **at least one** `g` goes over 0.95).
3. **Cost:** measure in seconds `KernelPCA(5, kernel="rbf", gamma=3).fit(X)` with `n` in `[500, 1000, 2000, 4000]` points of `make_moons(n, noise=0.15)`: save `times` (`pd.Series` indexed by `n`) and `exponent` = slope of the line `log(time) ~ log(n)` (with `np.polyfit`).

## Setup

Run this cell without changing it.

In [ ]:
import sys, pathlib
_root = next(p for p in [pathlib.Path.cwd(), *pathlib.Path.cwd().parents] if (p / "common").is_dir())
sys.path.insert(0, str(_root))

import warnings; warnings.filterwarnings("ignore")     # clean output: here the warnings are just noise
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
from common import datasets as ds
from common.verify import check, close, done, verify

import time
from sklearn.datasets import make_circles, make_moons
from sklearn.decomposition import PCA, KernelPCA
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import StratifiedKFold, cross_val_score
from sklearn.pipeline import make_pipeline

datasets = {"circles": make_circles(1000, noise=0.05, factor=0.4, random_state=0), "moons": make_moons(1000, noise=0.15, random_state=0)}
cv = StratifiedKFold(5, shuffle=True, random_state=0)

## Your work

In [ ]:
base, pca2 = {}, {}
for name, (X, y) in datasets.items():
    ...
pd.DataFrame({"logistic": base, "PCA(2)": pca2}).round(3)

In [ ]:
grid = {}
best = {}
min_k = {}

In [ ]:
times = pd.Series(dtype=float)
exponent = None

## Check

Run the cell when you think you are done. If a check fails, the message tells you what is wrong. If it fails a second time, a hint appears under the message.

In [ ]:
with verify(hint='`cross_val_score(make_pipeline(KernelPCA(k, kernel="rbf", gamma=g), LogisticRegression(max_iter=1000)), X, y, cv=cv).mean()`. Times: `t0 = time.time(); KernelPCA(5, kernel="rbf", gamma=3).fit(X); time.time() - t0`. Exponent: `np.polyfit(np.log(times.index), np.log(times.values), 1)[0]`.'):
    check(set(base) == {"circles", "moons"} and base["circles"] < 0.55 and pca2["circles"] < 0.55 and 0.8 < base["moons"] < 0.9, "circles: logistic and PCA(2) at chance (below 0.55); moons: the logistic reaches 0.85", "PCA(2) is a rotation: it changes nothing")
    check(all(grid[n].shape == (3, 3) for n in ("circles", "moons")) and grid["circles"].loc[5].max() > 0.99 and grid["moons"].loc[5].max() > 0.96, "KernelPCA with 5 components: circles over 0.99, moons over 0.96", "RBF kernel, k=5")
    check(best["circles"][0] >= 5 and min_k["circles"] == 5 and min_k["moons"] == 5, "5 components are needed (2 is not enough with any gamma)", "the smallest k with accuracy above 0.95")
    check(grid["circles"].loc[2].max() < 0.8 and grid["moons"].loc[2].max() < 0.95, "with 2 components KernelPCA does not solve the problem (circles below 0.8, moons below 0.95)", "two components are not enough")
    check(list(times.index) == [500, 1000, 2000, 4000] and times.is_monotonic_increasing and exponent > 1.3 and times[4000] > 5 * times[1000], "the time grows more than linearly with n (exponent over 1.3; 4000 points over 5 times slower than 1000)", "the kernel matrix is n x n")
done()

---
The commented solution is in `solutions/F_compare/F2_linear_pca_vs_kernel_pca.ipynb`. Open it after you pass the check, or after you have really tried.